# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarisAhmed786/FlyRank-Scalable-SEO-Ranking-ML-Agent/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import os
import subprocess
from pathlib import Path

REPO_DIR = "flyrank-ml-internship-starter"
REPO_URL = "https://github.com/HarisAhmed786/FlyRank-Scalable-SEO-Ranking-ML-Agent"

# Reset to Colab root
os.chdir("/content")

# Clone if missing
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

# Enter repo
os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())

print(
    "Dataset exists:",
    Path("data/raw/content_refresh_anonymized.csv").exists()
)

Working directory: /content/flyrank-ml-internship-starter
Dataset exists: True


## 1. My lane as an ML task (type)

This is a ranking/scoring task, not plain classification. The real decision is not only "is this page declining, yes/no" — it is "which pages should an editor review first when review time is limited?" A binary classifier can identify declining pages, but an editor needs an ordered queue of pages based on priority. Therefore, the output should be a priority score that ranks pages according to patterns associated with observed decline. Precision@K is an appropriate evaluation metric because it measures how many of the top-ranked pages are actually declining.

## 2. Target or proxy
The target is is_declining_label, created from the observed trend_direction field where:

trend_direction == "down"

This represents pages that experienced a historical decline in search impressions. However, it is not a direct measurement of whether a page truly needed a content refresh, so it should be treated as a proxy for refresh priority rather than a perfect ground-truth label. A page may decline because of factors such as seasonality, SERP changes, competition, or other external factors that a content refresh may not solve.

In [5]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

df["is_declining_label"].value_counts()

,count
is_declining_label,
1,16262
0,13738


## 3. Success metric
The primary success metric will be Precision@20, with Precision@50 as a secondary evaluation.

Precision@K measures the percentage of pages in the top-K recommended pages that are actually declining. This matches the real workflow because SEO editors usually have limited time and review the highest-priority pages first. A successful ranking system should place truly declining pages near the top of the queue rather than optimizing overall accuracy across all pages.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe
The unit of analysis is one page/content asset. Each row represents a single page tracked over a period of time. This matches the real business action because an SEO editor reviews individual pages and decides whether they should be refreshed.


In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

print(df.shape)

df[
    [
        "impressions_90d",
        "days_since_last_update",
        "avg_position",
        "ctr",
        "trend_direction"
    ]
].head()

(30000, 45)


,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
0,3803,20,10.6,0.76,down
1,15320,25,20.3,0.05,down
2,12581,20,36.5,0.09,down
3,11751,22,6.2,0.49,stable
4,19140,14,44.0,0.13,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule such as identifying pages that are both old and have low traffic can provide a simple baseline, but it cannot combine multiple signals effectively. A page that is not very old may still need attention if other signals, such as ranking position, impressions, or engagement, indicate decline.

The data suggests that page performance depends on a combination of factors rather than one simple threshold. A machine learning model can learn these patterns by combining multiple signals and assigning a ranking score. This makes it more suitable than a manually written rule when the relationships between signals are complex.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.